**MRP**

**核心假设：某时刻的状态只取决于上一时刻的状态**

In [1]:
import numpy as np

np.random.seed(0)
P = [
    [0.9, 0.1, 0.0, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.5, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.6, 0.0, 0.4],
    [0.0, 0.0, 0.0, 0.0, 0.3, 0.7],
    [0.0, 0.2, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
]
P = np.array(P)

rewards = [-1, -2, -2, 10, 1, 0]   # 奖励函数
gamma = 0.5                        # 折扣因子 γ（用于表示远期利益的不确定性）

def compute_return(start_index, chain, gamma):
    G = 0
    for i in reversed(range(start_index, len(chain))):
        G = gamma * G + rewards[chain[i]-1]
    return G

**计算状态价值函数**
$$V(s) = r(s) + \gamma \sum_{s'\in\mathcal{S}} P(s' \mid s) V(s')$$

展开为向量形式：
$$
\begin{bmatrix}
V(s_1) \\
V(s_2) \\
\cdots \\
V(s_n)
\end{bmatrix}
=
\begin{bmatrix}
r(s_1) \\
r(s_2) \\
\cdots \\
r(s_n)
\end{bmatrix}
+
\gamma
\begin{bmatrix}
P(s_1\mid s_1) & P(s_2\mid s_1) & \cdots & P(s_n\mid s_1) \\
P(s_1\mid s_2) & P(s_2\mid s_2) & \cdots & P(s_n\mid s_2) \\
\cdots & \cdots & \cdots & \cdots \\
P(s_1\mid s_n) & P(s_2\mid s_n) & \cdots & P(s_n\mid s_n)
\end{bmatrix}
\begin{bmatrix}
V(s_1) \\
V(s_2) \\
\cdots \\
V(s_n)
\end{bmatrix}
$$

即：
$$
\begin{aligned}
\mathcal{V} &= \mathcal{R} + \gamma \mathcal{P}\mathcal{V} \\
(\mathcal{I} - \gamma \mathcal{P})\mathcal{V} &= \mathcal{R} \\
\mathcal{V} &= (\mathcal{I} - \gamma \mathcal{P})^{-1}\mathcal{R}
\end{aligned}
$$

该算法的时间复杂度为 $O(n^3)$，所以只适合求解规模较小的马尔可夫过程

In [2]:
def compute(P, rewards, gamma):
    """MRP贝尔曼方程解析解"""

    states_num = P.shape[0]
    rewards = np.array(rewards).reshape((-1, 1))
    I = np.eye(states_num)

    value = np.dot(np.linalg.inv(I - gamma * P), rewards)
    return value

**MDP**

现实生活中，$P(s' \mid s)$ 不单单只和状态有关，还和在 s 下执行的动作 a 相关\
同样，奖励函数也和 a 相关

引入动作价值函数：
$$
\begin{aligned}
Q^\pi(s,a) &= \mathbb{E}_\pi\big[R_t + \gamma Q^\pi(S_{t+1},A_{t+1}) \mid S_t = s, A_t = a\big] \\
&= r(s,a) + \gamma \sum_{s'\in\mathcal{S}} P(s' \mid s,a) \sum_{a'\in\mathcal{A}} \pi(a' \mid s') Q^\pi(s',a')
\end{aligned}
$$
则：
$$
V^\pi(s) = \sum_{a\in\mathcal{A}} \pi(a \mid s) Q^\pi(s,a)
$$

**计算策略 $\pi$ 下的状态价值函数**

对于某一个状态，将奖励函数和状态转移函数根据动作的概率进行加权：
$$
\begin{aligned}
r'(s) &= \sum_{a\in\mathcal{A}} \pi(a \mid s) r(s,a) \\
P'(s' \mid s) &= \sum_{a\in\mathcal{A}} \pi(a \mid s) P(s' \mid s,a)
\end{aligned}
$$
运用 MRP 的公式，即可计算出每个状态下的状态价值函数（同样也只适用于状态数较小的情况）

**蒙特卡洛方法**

原理：大数定律
$$
\begin{aligned}
N(s) &\leftarrow N(s)+1 \\
V(s) &\leftarrow V(s) + \frac{1}{N(s)}\big(G - V(s)\big)
\end{aligned}
$$


In [3]:
S = ["s1", "s2", "s3", "s4", "s5"]  # 状态集合
A = ["保持 s1", "前往 s1", "前往 s2", "前往 s3", "前往 s4", "前往 s5", "概率前往"] # 动作集合

# 状态转移概率
P = {
    "s1-保持 s1-s1":1.0, "s1-前往 s2-s2":1.0,
    "s2-前往 s1-s1":1.0, "s2-前往 s3-s3":1.0,
    "s3-前往 s4-s4":1.0, "s3-前往 s5-s5":1.0,
    "s4-前往 s5-s5":1.0, "s4-概率前往-s2":0.2,
    "s4-概率前往-s3":0.4, "s4-概率前往-s4":0.4,
}
# 奖励字典
R = {
    "s1-保持 s1":-1, "s1-前往 s2":0,
    "s2-前往 s1":-1, "s2-前往 s3":-2,
    "s3-前往 s4":-2, "s3-前往 s5":0,
    "s4-前往 s5":10, "s4-概率前往":1,
}

gamma = 0.5 # 折扣因子
MDP = (S, A, P, R, gamma)

# 拼接字符串："状态-动作" 作为字典key
def join(str1, str2):
    return str1 + '-' + str2

In [10]:
def sample(MDP, Pi, timestep_max, number):
    ''' 采样函数, 策略Pi, 限制最长时间步timestep_max, 总共采样序列数number '''

    S, A, P, R, gamma = MDP
    episodes = []
    for _ in range(number):
        episode = []
        timestep = 0
        # 随机选起点：除终止状态s5以外
        s = S[np.random.randint(4)]
        # 未到终止状态、且没超过最大步数则持续交互
        while s != "s5" and timestep <= timestep_max:
            timestep += 1

            # ==========1. 根据策略 Pi 采样动作 a ==========
            rand = np.random.rand() # 生成一个 0~1 之间均匀随机数
            temp = 0 # 概率累加器
            for a_opt in A: 
                temp += Pi.get(join(s, a_opt), 0) # 状态s下选择动作a_opt的概率，若没有定义则概率为 0
                if temp > rand:
                    a = a_opt
                    r = R.get(join(s, a), 0)
                    break
            # ==========2. 根据转移概率P采样下一个状态 s_next ==========
            rand = np.random.rand()
            temp = 0
            for s_opt in S:
                temp += P.get(join(join(s, a), s_opt), 0)
                if temp > rand:
                    s_next = s_opt
                    break

            episode.append((s, a, r, s_next))
            s = s_next
        episodes.append(episode)
        
    return episodes

In [11]:
def MC(episodes, V, N, gamma):
    for episode in episodes:
        G = 0
        # 从后向前反向遍历轨迹
        for i in range(len(episode)-1, -1, -1):
            (s, a, r, s_next) = episode[i]
            G = r + gamma * G
            N[s] = N[s] + 1
            V[s] = V[s] + (G - V[s]) / N[s]

In [12]:
Pi_1 = {
    "s1-保持 s1":0.5, "s1-前往 s2":0.5,
    "s2-前往 s1":0.5, "s2-前往 s3":0.5,
    "s3-前往 s4":0.5, "s3-前往 s5":0.5,
    "s4-前往 s5":0.5, "s4-概率前往":0.5,
}

timestep_max = 20
episodes = sample(MDP, Pi_1, timestep_max, 1000)
gamma = 0.5

# 初始化价值V、计数N
V = {"s1":0, "s2":0, "s3":0, "s4":0, "s5":0}
N = {"s1":0, "s2":0, "s3":0, "s4":0, "s5":0}
# 蒙特卡洛估计状态价值
MC(episodes, V, N, gamma)
print("使用蒙特卡洛方法计算 MDP 的状态价值为\n", V)

使用蒙特卡洛方法计算 MDP 的状态价值为
 {'s1': -1.2037918620639356, 's2': -1.6951508779111135, 's3': 0.5038593021903446, 's4': 6.0805818654863195, 's5': 0}


**最优策略**

在每一个状态下，都选取能最大化该状态下的价值的动作